In [1]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.express as px
import os
import sys

: 

: 

In [ ]:
sys.path.insert(1, '..')
from SepsisPkg.config import SEPSIS_CONFIG

In [5]:
RAW_DATA_PATH = "../data/raw_data"
files = sorted(os.listdir(RAW_DATA_PATH))

# Confirm that the schemas match over different csv files
schema = {}
pats = {}
for file in files:
    if not file.endswith('.csv'):
        print(f"{file} can not be read by pl.scan_csv(...)")
    fpath = os.path.join(RAW_DATA_PATH, file) 
    df = pl.scan_csv(
        fpath
    )
    uniq_pats = df.select("PAT_ENC_CSN_ID").unique()
    schema[file] = uniq_pats

In [6]:
pat_diag = schema['Diagnosis Audit - Nov 2024 - Oct 2025.csv'].collect_async()
pat_flow = schema['Flowsheets - Nov 2024 - Oct 2025.csv'].collect_async()
pat_labord = schema['Lab Order Results - Nov 2024 - Oct 2025.csv'].collect_async()
pat_medadmin = schema['Medication Admin - Nov 2024 - Oct 2025.csv'].collect_async()
pat_medord = schema['Medication Orders - Nov 2024 - Oct 2025.csv'].collect_async()
pat_proc = schema['Procedure Orders - Nov 2024 - Oct 2025.csv'].collect_async()

In [7]:
pat_diag = pat_diag.result.result()
pat_flow = pat_flow.result.result()
pat_labord = pat_labord.result.result()
pat_medadmin = pat_medadmin.result.result()
pat_medord = pat_medord.result.result()
pat_proc = pat_proc.result.result()

In [8]:
len(pat_flow), len(pat_labord), len(pat_diag), len(pat_medadmin), len(pat_medord), len(pat_proc)

(3930, 3927, 3930, 3923, 3927, 3922)

In [9]:
uniq_pats = set(pat_flow['PAT_ENC_CSN_ID']).intersection(set(pat_diag['PAT_ENC_CSN_ID'])).\
    intersection(set(pat_labord["PAT_ENC_CSN_ID"])).intersection(set(pat_medadmin["PAT_ENC_CSN_ID"])).\
        intersection(set(pat_medord["PAT_ENC_CSN_ID"])).intersection(set(pat_proc['PAT_ENC_CSN_ID']))

In [10]:
df_t = []
for file in files:
    if not file.endswith('.csv'):
        print(f"{file} can not be read by pl.scan_csv(...)")
    fpath = os.path.join(RAW_DATA_PATH, file) 
    df = pl.scan_csv(
        fpath, null_values=["NULL", 'null', "Null"], infer_schema_length=int(1e7)
    )
    # df_t.append(df.filter(pl.col("PAT_ENC_CSN_ID").is_in(rpats)).collect_async())
    df_t.append(df.collect_async())

In [11]:
df_data = []
for dd in df_t:
    if dd.result.done():
        df_data.append(dd.result.result())
len(df_data)

6

In [12]:
for i, df_i in enumerate(df_data):
    df_i = df_i.with_columns(
        pl.col("Event_ID").cast(pl.Int64)
    )
    if 'Event_Datetime' in df_i.columns:
        df_i = df_i.rename({"Event_Datetime":"Event_DateTime"})
    df_data[i] = df_i

df_all = pl.concat(df_data, how='vertical')

In [13]:
df_all = df_all.sort(by=["PAT_ENC_CSN_ID", "Event_DateTime"])

In [14]:
if df_all.schema['Admit_Time'] in [pl.Utf8, pl.String]:
    df_all = df_all.with_columns(
        pl.col("Admit_Time").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.f")
    )

if df_all.schema['Event_DateTime'] in [pl.Utf8, pl.String]:
    df_all = df_all.with_columns(
        pl.col("Event_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.f")
    )

if df_all.schema['Pt_Arrival'] in [pl.Utf8, pl.String]:
    df_all = df_all.with_columns(
        pl.col("Pt_Arrival").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.f")
    )


In [15]:
drop_cols = []
for col in ["DateADmitKey", "Chief_Complaint", "Event_ID"]:
    if col in df_all.columns:
        drop_cols.append(col)

df_all = df_all.drop(drop_cols)

In [16]:
df_all = df_all.with_columns(
    event_full_name=pl.col("Type")+"__"+pl.col("EVENT_NAME")+"__"+pl.col("Event_Grouper")
)

In [17]:
df_all.write_parquet("../data/preprocessed_all_data.parquet")


# Start reading preprocessed data from here

In [4]:
df_all = pl.read_parquet("../data/preprocessed_all_data.parquet")

In [32]:
# df_all.filter(
#    expr_culture_orders 
# )['event_full_name'].unique().to_list()
df_all.filter(
    expr_culture_orders
).shape

(59985, 25)

In [65]:
df_all.filter(pl.col("Event_Grouper").str.contains("Vent"))['Event_Grouper'].unique()

Event_Grouper
str
"""Vent On/Off"""


In [42]:
sepsis_expr_list = []
antisepsis_expr_list = []

organism_keywords = [
    "staph", "strep", "coli", "klebsiella", "pseudomonas", "enterococcus", 
    "candida", "aspergillus", "bacteroides", "clostridium", "serratia", 
    "proteus", "acinetobacter", "citrobacter", "enterobacter",
    "detected", "positive" # Keep generic terms just in case
]

expr_culture_positive_result = (
    # 1. It must be a Lab Result
    (pl.col("event_full_name").str.to_lowercase().str.contains("result")) &
    
    # 2. It must mention a culture source
    (pl.col("event_full_name").str.to_lowercase().str.contains("culture|bcid")) &
    
    # 3. It must match a specific bug OR say "Detected"
    (pl.col("event_full_name").str.to_lowercase().str.contains_any(organism_keywords)) &
    
    # 4. EXCLUDE contaminants if necessary (e.g., "contamination likely")
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("contamina"))
)


sepsis_expr_list.append(expr_culture_positive_result)

# The "Anchor" for your Rolling Window
expr_culture_orders = (
    # Look for the ORDER or PROCEDURE
    (pl.col("event_full_name").str.to_lowercase().str.contains("order")) &
    (pl.col("event_full_name").str.to_lowercase().str.contains("culture")) &
    
    # Filter for the actual REQUEST (Procedure), not the Result
    (pl.col("event_full_name").str.to_lowercase().str.contains("procedure"))
)

sepsis_expr_list.append(expr_culture_orders)


expr_antibiotics_gold_standard = (
    (pl.col("Event_Grouper") == "Antibiotics") & 
    
    # 1. INCLUSION: Must be a systemic route
    (pl.col("event_full_name").str.to_lowercase().str.contains(r"\biv\b|intravenous|infusion|injection|push|piggy back")) &
    
    # 2. EXCLUSION: The "Not Sepsis" Filter
    ~(pl.col("event_full_name").str.to_lowercase().str.contains(
        r"dialysis|"        # Maintenance fluids
        r"heparin|"         # Line flushes
        r"lock solution|"   # Catheter cleaning
        r"chemo|"           # Cancer treatment
        r"rubicin|"         # Specific chemo agents (Doxorubicin, etc.)
        r"epoch|"           # Chemo cocktail
        r"intravitreal|"    # Eye injections (Local)
        r"intrapleural|"    # Lung cavity (Local/Mechanical)
        r"alteplase|"       # Clot busters
        r"activase"         # Clot busters
    ))
)
sepsis_expr_list.append(expr_antibiotics_gold_standard)

expr_incision_debr = (
    (pl.col("event_full_name").str.to_lowercase().str.contains("procedure"))&
    (
        (pl.col("event_full_name").str.to_lowercase().str.contains("incision")) | 
        (pl.col("event_full_name").str.to_lowercase().str.contains("debrid"))
    )
)

sepsis_expr_list.append(expr_incision_debr)

expr_procedure_drain = (
    (pl.col("event_full_name").str.to_lowercase().str.contains("procedure"))&
    (
        (pl.col("event_full_name").str.to_lowercase().str.contains("drain")) 
    )
)
sepsis_expr_list.append(expr_procedure_drain)

expr_dopamine = (pl.col("event_full_name").str.to_lowercase().str.contains("dopamine"))
sepsis_expr_list.append(expr_dopamine)

expr_norepinephrine = (pl.col("event_full_name").str.to_lowercase().str.contains("norepinephrine"))
sepsis_expr_list.append(expr_norepinephrine)

expr_vasopressin = (
    (pl.col("event_full_name").str.to_lowercase().str.contains("vasopressin"))&
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("__other"))
)
sepsis_expr_list.append(expr_vasopressin)

expr_phenylephrine = (
    (pl.col("event_full_name").str.to_lowercase().str.contains("phenylephrine"))&
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("push dose"))&
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("10 mg"))&
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("1 mg"))
)
sepsis_expr_list.append(expr_phenylephrine)

chest_xray_ct = [
    "XR CHEST 1 VIEW",
    "XR CHEST 2 VIEWS",
    "XR OUTSIDE STUDY CHEST INTERPRETATION REQUEST",
    "CT CHEST WO IV CONTRAST",
    "CT CHEST W IV CONTRAST",
    "CT CHEST W AND WO IV CONTRAST",
    "CT CHEST HIGH RESOLUTION",
    "CT CHEST LIMITED",
    "CT OUTSIDE STUDY CHEST INTERPRETATION REQUEST",
    "US CHEST OR UPPER BACK SOFT TISSUES",
    "US CHEST CAVITY"
]

expr_chest_xray = pl.col("EVENT_NAME").is_in(chest_xray_ct)
sepsis_expr_list.append(expr_chest_xray)

expr_temp = (
    pl.col("event_full_name").str.to_lowercase().str.contains("temp")&
    pl.col("event_full_name").str.to_lowercase().str.contains("flowsheet")
)
sepsis_expr_list.append(expr_temp)

expr_respiration = (
    pl.col("event_full_name").str.to_lowercase().str.contains("respirations")&
    pl.col("event_full_name").str.to_lowercase().str.contains("flowsheet")
)
sepsis_expr_list.append(expr_respiration)

expr_o2 = (pl.col("EVENT_NAME")=="CPM S24 R INV FLOW (L/MIN) (OXYGEN THERAPY)")
sepsis_expr_list.append(expr_o2)

expr_pulse = (pl.col("EVENT_NAME")=="PULSE")
sepsis_expr_list.append(expr_pulse)

expr_bp = (pl.col("EVENT_NAME")=="BLOOD PRESSURE")
sepsis_expr_list.append(expr_bp)

expr_cancer = (pl.col('event_full_name').str.to_lowercase().str.contains("cancer"))
antisepsis_expr_list.append(expr_cancer)

expr_pulm_embo = (pl.col('event_full_name').str.to_lowercase().str.contains("pulmonary embolism"))
antisepsis_expr_list.append(expr_pulm_embo)

expr_bun = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("bun"))
)
sepsis_expr_list.append(expr_bun)

expr_plat = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("platelet"))
)
sepsis_expr_list.append(expr_plat)

expr_wbc = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("wbc"))
)
sepsis_expr_list.append(expr_wbc)

expr_bilirubin = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("bili"))
)
sepsis_expr_list.append(expr_bilirubin)

expr_lactate = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("lact"))
)
sepsis_expr_list.append(expr_lactate)


# expr_antibiotics_refined = (
#     (pl.col("Event_Grouper") == "Antibiotics") & 
#     (pl.col("event_full_name").str.to_lowercase().str.contains(" iv |intravenous|infusion|injection")) &
#     # EXCLUSIONS
#     ~(pl.col("event_full_name").str.to_lowercase().str.contains("lock solution")) & 
#     ~(pl.col("event_full_name").str.to_lowercase().str.contains("dialysis")) # <--- NEW CRITICAL FIX
# )
# sepsis_expr_list.append(expr_antibiotics_refined)

# 2. FUNGAL CULTURES (Positive Detection)
# We look for the "Bug" names (from your previous list) or explicit "Positive" text
fungal_keywords = [
    "candida", "aspergillus", "cryptococcus", "histoplasma", 
    "blastomyces", "coccidioides", "pneumocystis"
]
exclusion_terms = [
    "candidate",       # Fixes "Transplant candidate"
    "prophylaxis",     # Fixes "Need for prophylaxis"
    "screening",       # Common in admission events
    "history of",      # Historical, not active
    "exposure to"      # Contact, not infection
]

# Refined Logic
expr_fungal_positive = (
    # 1. Must match one of the fungal keywords
    (pl.col("event_full_name").str.to_lowercase().str.contains_any(fungal_keywords)) &
    
    # 2. Must NOT match any of the exclusion terms
    ~(pl.col("event_full_name").str.to_lowercase().str.contains_any(exclusion_terms)) &
    
    # 3. FIX: Enforce "Candida" is its own word (exclude "Candidate")
    # This regex means: "If 'candida' appears, check that it is NOT followed by 'te'"
    # Or simply rely on the exclusion list above, which is cleaner.
    
    # 4. Optional: Explicitly exclude 'candidate' via regex if not using contains_any list above
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("candidate")))

sepsis_expr_list.append(expr_fungal_positive)

In [26]:
expr_antibiotics_safe = (
    (pl.col("EVENT_Grouper") == "Antibiotics") & 
    (pl.col("event_full_name").str.to_lowercase().str.contains(r"\biv\b|intravenous|infusion|injection")) &
    
    # EXCLUSIONS (Critical Safety Nets)
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("lock solution")) & 
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("dialysis")) &  # <--- FIXES THE FALSE FLUIDS
    ~(pl.col("event_full_name").str.to_lowercase().str.contains("heparin"))      # <--- Common additive, not abx
)

# ---------------------------------------------------------
# 2. UPDATED "ANTI-SEPSIS" MIMIC LOGIC
# ---------------------------------------------------------

# A. Candidates (Conditions that look like Sepsis)
# Added: rhabdomyolysis, hypercalcemia
expr_mimic_candidate = (
    pl.col("event_full_name").str.to_lowercase().str.contains(
        r"cancer|pulmonary embolism|metast|neoplasm|tumor|malignan|lysis|neuroleptic|rhabdomyolysis|hypercalcemia"
    )
)

# B. Noise (Things that are irrelevant)
# Added: "abnormal finding", "fatigue"
expr_mimic_noise = (
    pl.col("event_full_name").str.to_lowercase().str.contains(
        r"history|screening|risk|anxiety|pain|candidate|encounter|suspected|prophylaxis|benign|uncertain|elevated|psa|abnormal|fatigue"
    ) |
    # Whole word abbreviations for history/status post
    pl.col("event_full_name").str.to_lowercase().str.contains(r"\bhx\b|\bh/o\b|\bs/p\b")
)

# C. The "Actually Sepsis" Exception
# Do NOT filter these out, even if they mention "Dialysis" or "Malignant"
expr_is_actually_sepsis = (
    pl.col("event_full_name").str.to_lowercase().str.contains("septic") |
    pl.col("event_full_name").str.to_lowercase().str.contains("peritonitis") |
    pl.col("event_full_name").str.to_lowercase().str.contains("empyema") | # Pus in lung
    pl.col("event_full_name").str.to_lowercase().str.contains("abscess") |
    (
        pl.col("event_full_name").str.to_lowercase().str.contains("infection") &
        pl.col("event_full_name").str.to_lowercase().str.contains("dialysis|graft|catheter")
    )
)

# D. Final Flag
expr_anti_sepsis = (
    expr_mimic_candidate &
    ~expr_mimic_noise &
    ~expr_is_actually_sepsis
)

# Sepsis detection logic

In [6]:
lf = df_all.lazy()

In [16]:
lf.schema

OrderedDict([('PAT_ENC_CSN_ID', Int64),
             ('PAT_MRN_ID', Int64),
             ('PAT_ID', String),
             ('Ethnicity', String),
             ('FirstRace', String),
             ('Sex', String),
             ('Coverage_Financial_Class_Grouper', String),
             ('Pt_Arrival', Datetime(time_unit='ns', time_zone=None)),
             ('Admit_Time', Datetime(time_unit='ns', time_zone=None)),
             ('DateAdmitKey', Int64),
             ('Patient_Age', Float64),
             ('Chief_Complaint_All', String),
             ('Before_Admit_YN', String),
             ('Admitted_From_ED', String),
             ('LOS_Days', Int64),
             ('Hours_in_ED', Int64),
             ('Event_DateTime', Datetime(time_unit='ns', time_zone=None)),
             ('Type', String),
             ('EVENT_NAME', String),
             ('Event_Grouper', String),
             ('Result_Flag', String),
             ('MEAS_VALUE', String),
             ('Sepsis_Category', String),
         

In [43]:
lf_abx = lf.filter(
    expr_antibiotics_gold_standard
).select(
    "PAT_ENC_CSN_ID",
    pl.col("Event_DateTime").alias("ABX_Time")
).sort("ABX_Time")

lf_culture = lf.filter(
    expr_culture_orders 
).select(
    "PAT_ENC_CSN_ID",
    pl.col("Event_DateTime").alias("culture_Time")
).sort("culture_Time")

lf_suspected_backward = lf_abx.join_asof(
    lf_culture,
    left_on="ABX_Time",
    right_on="culture_Time",
    by="PAT_ENC_CSN_ID",
    strategy="backward",
    tolerance="24h"
).filter(pl.col("culture_Time").is_not_null())

lf_suspected_forward = lf_abx.join_asof(
    lf_culture,
    left_on="ABX_Time",
    right_on="culture_Time",
    by="PAT_ENC_CSN_ID",
    strategy="forward",
    tolerance="72h"
).filter(pl.col("culture_Time").is_not_null())

# Combine matches and pick the EARLIEST of the two times
df_suspected_infection = (
    pl.concat([lf_suspected_backward, lf_suspected_forward])
    .unique()
    .with_columns(
        # Sepsis Time Zero = Min(Abx Time, Culture Time)
        pl.min_horizontal(["ABX_Time", "culture_Time"]).alias("Suspected_Infection_Time")
    )
    # Get the FIRST episode per encounter
    .group_by("PAT_ENC_CSN_ID")
    .agg(pl.col("Suspected_Infection_Time").min())
    .collect()
)

print(f"Total Sepsis Episodes Found: {df_suspected_infection.height}")

Total Sepsis Episodes Found: 3595


In [44]:
df_suspected_infection

PAT_ENC_CSN_ID,Suspected_Infection_Time
i64,datetime[ns]
727200338,2025-05-11 16:58:08
714738435,2024-11-08 15:11:00
723798812,2025-03-22 00:11:37
728815638,2025-06-04 21:27:00
721712678,2025-02-18 17:10:22
…,…
722086381,2025-02-24 22:07:15
724661177,2025-04-04 05:53:00
716116344,2024-11-18 12:33:40


In [45]:
df_all.schema

OrderedDict([('PAT_ENC_CSN_ID', Int64),
             ('PAT_MRN_ID', Int64),
             ('PAT_ID', String),
             ('Ethnicity', String),
             ('FirstRace', String),
             ('Sex', String),
             ('Coverage_Financial_Class_Grouper', String),
             ('Pt_Arrival', Datetime(time_unit='ns', time_zone=None)),
             ('Admit_Time', Datetime(time_unit='ns', time_zone=None)),
             ('DateAdmitKey', Int64),
             ('Patient_Age', Float64),
             ('Chief_Complaint_All', String),
             ('Before_Admit_YN', String),
             ('Admitted_From_ED', String),
             ('LOS_Days', Int64),
             ('Hours_in_ED', Int64),
             ('Event_DateTime', Datetime(time_unit='ns', time_zone=None)),
             ('Type', String),
             ('EVENT_NAME', String),
             ('Event_Grouper', String),
             ('Result_Flag', String),
             ('MEAS_VALUE', String),
             ('Sepsis_Category', String),
         

In [48]:
df_suspected_infection_all = df_suspected_infection.join(
    df_all.select("PAT_ENC_CSN_ID", "event_full_name", "Admitted_From_ED", "Admit_Time", "Pt_Arrival", "Patient_Age", "Sepsis_Category", "FLAG_DEATH_CLARITY"),
    on="PAT_ENC_CSN_ID"
)

In [53]:
df_all['Event_Grouper'].unique().to_list()

['Respiratory Assessment',
 'Other',
 'Pain Scale',
 'Vent On/Off',
 'NEUTROS ABS',
 'MONOS PCT',
 'IMMATURE GRAN PCT',
 'Blood Urea Nitrogen',
 'PaO2',
 'Anti-Rheumatics',
 'RDW',
 'Braden Scale',
 'NEUTROS PCT',
 'Skin Assessment',
 'Antiemetics',
 'Body Fluid Cultures',
 'IMMATURE GRAN ABS',
 'Lymphocytes Percent',
 'PH ART',
 'MCHC',
 'Absolute Lymphocytes',
 'Dobutamine',
 'Creatinine',
 'nRBC',
 'Pulse',
 'Antibiotics',
 'Respirations',
 'Sepsis',
 'Oxygen Delivery',
 'Bilirubin',
 'MONOS ABS',
 'Imaging',
 'Epinephrine',
 'Weight',
 'RBC',
 'FiO2',
 'Chloride',
 'Blood Pressure',
 'AST',
 'Severe Sepsis',
 'PROCALCITONIN',
 'Dopamine',
 'Platelet Count',
 'Phenylephrine',
 'Neurological Assessment',
 'Temperature',
 'BMI',
 'UTI',
 'Gastrointestinal Assessment',
 'Vasopressin',
 'Urine Output',
 'EKG',
 'Analgesic Non-Narcotics',
 'Glasgow Coma Score',
 'O2 Flow Rate',
 'WBC',
 'ALBUMIN',
 'Lactate',
 'Genitourinary Assessment',
 'Sodium',
 'Septic Shock',
 'Norepinephrine']

In [ ]:
df_all.filter(
    pl.col("Event_Grouper")
)

In [51]:
df_suspected_infection_all.group_by("PAT_ENC_CSN_ID").agg(
    pl.col("Admit_Time").first(),
    pl.col("Pt_Arrival").first(),
    pl.col("Patient_Age").first(),
    pl.col("Suspected_Infection_Time").first(),
    pl.col("Sepsis_Category").first(),
    pl.col("Admitted_From_ED").first(),
    pl.col("FLAG_DEATH_CLARITY").first(),
).with_columns(
    (pl.col("Suspected_Infection_Time")-pl.col("Pt_Arrival")).dt.total_minutes().alias('total_min_between_arrival_to_suspicion')
) 

PAT_ENC_CSN_ID,Admit_Time,Pt_Arrival,Patient_Age,Suspected_Infection_Time,Sepsis_Category,Admitted_From_ED,FLAG_DEATH_CLARITY,total_min_between_arrival_to_suspicion
i64,datetime[ns],datetime[ns],f64,datetime[ns],str,str,i64,i64
726578627,2025-05-01 19:19:00,2025-05-01 12:08:00,68.577686,2025-05-01 12:39:03,"""POA-3""","""Yes""",0,31
720421146,2025-01-31 07:14:00,2025-01-30 07:18:00,72.032854,2025-01-30 16:56:00,"""POA-1""","""Yes""",0,578
721712678,2025-02-18 22:27:00,2025-02-18 16:01:00,76.142368,2025-02-18 17:10:22,"""POA-3""","""Yes""",0,69
730727172,2025-07-14 11:54:08,2025-07-14 07:54:27,36.402464,2025-07-14 12:53:00,"""POA-1""","""No""",0,298
714738435,2024-10-29 20:35:48,2024-10-29 10:32:50,56.862422,2024-11-08 15:11:00,"""NPOA-3""","""No""",0,14678
…,…,…,…,…,…,…,…,…
733325691,2025-08-13 20:12:00,2025-08-13 13:06:00,22.631074,2025-08-13 13:19:46,"""POA-3""","""Yes""",0,13
729388706,2025-06-12 21:55:00,2025-06-12 15:43:00,58.195756,2025-06-12 16:01:49,"""POA-3""","""Yes""",0,18
731568833,2025-07-18 14:15:00,2025-07-17 13:31:00,84.418891,2025-07-17 14:10:05,"""POA-2""","""Yes""",0,39


In [55]:
# 1. PREPARE DEMOGRAPHICS (One row per encounter)
# We take the main log and select distinct demographic info
df_demo = df_all.select([
    "PAT_ENC_CSN_ID", "Pt_Arrival", "Admit_Time", "Admitted_From_ED", 
    "Patient_Age", "FLAG_DEATH_CLARITY", "Sepsis_Category"
PAT_ENC_CSN_ID	Suspected_Infection_Time	Pt_Arrival	Admit_Time	Admitted_From_ED	Patient_Age	FLAG_DEATH_CLARITY	Sepsis_Category	Hours_Arrival_to_Suspicion	Is_Hospital_Acquired	Age_Group
i64	datetime[ns]	datetime[ns]	datetime[ns]	str	f64	i64	str	i64	bool	str
727200338	2025-05-11 16:58:08	2025-05-11 15:46:00	2025-05-12 00:42:00	"Yes"	75.997262	0	"POA-1"	1	false	"Elderly"
714738435	2024-11-08 15:11:00	2024-10-29 10:32:50	2024-10-29 20:35:48	"No"	56.862422	0	"NPOA-3"	244	true	"Adult"
723798812	2025-03-22 00:11:37	2025-03-21 23:42:00	2025-03-22 15:03:00	"Yes"	23.195071	0	"POA-1"	0	false	"Adult"
728815638	2025-06-04 21:27:00	2025-06-04 19:05:00	2025-06-05 23:48:00	"Yes"	55.137577	0	"POA-1"	2	false	"Adult"
721712678	2025-02-18 17:10:22	2025-02-18 16:01:00	2025-02-18 22:27:00	"Yes"	76.142368	0	"POA-3"	1	false	"Elderly"
…	…	…	…	…	…	…	…	…	…	…
722086381	2025-02-24 22:07:15	2025-02-24 17:07:00	2025-02-24 21:23:00	"Yes"	80.052019	0	"NPOA-3"	5	false	"Elderly"
724661177	2025-04-04 05:53:00	2025-04-03 22:21:00	2025-04-04 17:35:00	"Yes"	72.402464	0	"POA-3"	7	false	"Elderly"
716116344	2024-11-18 12:33:40	2024-11-18 12:29:00	2024-11-19 22:06:00	"Yes"	68.150581	0	"POA-1"	0	false	"Elderly"
716574989	2024-11-27 14:55:18	2024-11-25 14:11:00	2024-11-26 22:29:00	"Yes"	85.281314	0	"POA-3"	48	false	"Elderly"
717381524	2024-12-10 08:29:00	2024-12-09 18:22:00	2024-12-10 17:08:00	"Yes"
]).unique(subset=["PAT_ENC_CSN_ID"])

# 2. JOIN WITH YOUR SEPSIS COHORT
df_analysis = df_suspected_infection.join(
    df_demo, on="PAT_ENC_CSN_ID", how="left"
)

# 3. CALCULATE METRICS
df_insights = df_analysis.with_columns(
    # A. Time to Treatment (in Hours)
    ((pl.col("Suspected_Infection_Time") - pl.col("Pt_Arrival")).dt.total_hours())
    .alias("Hours_Arrival_to_Suspicion"),
    
    # B. Hospital Acquired Flag (>48 hours after admission)
    (pl.col("Suspected_Infection_Time") > (pl.col("Admit_Time") + pl.duration(hours=48)))
    .alias("Is_Hospital_Acquired"),
    
    # C. Age Groups
    pl.when(pl.col("Patient_Age") < 18).then(pl.lit("Pediatric"))
      .when(pl.col("Patient_Age") < 65).then(pl.lit("Adult"))
      .otherwise(pl.lit("Elderly")).alias("Age_Group")
)

# ---------------------------------------------------------
# GENERATE SUMMARY STATS FOR THE PI
# ---------------------------------------------------------

# Metric 1: Mortality Rate by Age
print("Mortality by Age Group:")
print(df_insights.group_by("Age_Group").agg(
    pl.count().alias("Total_Cases"),
    pl.col("FLAG_DEATH_CLARITY").sum().alias("Deaths"),
    (pl.col("FLAG_DEATH_CLARITY").mean() * 100).alias("Mortality_Rate_%")
).sort("Mortality_Rate_%", descending=True))

# Metric 2: Community vs Hospital Acquired
print("\nCommunity vs Hospital Acquired Sepsis:")
print(df_insights.group_by("Is_Hospital_Acquired").agg(
    pl.count().alias("Total_Cases"),
    (pl.col("FLAG_DEATH_CLARITY").mean() * 100).alias("Mortality_Rate_%")
))

# Metric 3: The "Missed Coding" Opportunity
# How many of OUR sepsis patients do NOT have a hospital Sepsis Label?
missed_coding = df_insights.filter(pl.col("Sepsis_Category").is_null()).height
print(f"\nPotential Uncoded Sepsis Cases: {missed_coding} out of {df_insights.height}")

Mortality by Age Group:


/tmp/ipykernel_183242/4293122646.py:36: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("Total_Cases"),


shape: (3, 4)
┌───────────┬─────────────┬────────┬──────────────────┐
│ Age_Group ┆ Total_Cases ┆ Deaths ┆ Mortality_Rate_% │
│ ---       ┆ ---         ┆ ---    ┆ ---              │
│ str       ┆ u32         ┆ i64    ┆ f64              │
╞═══════════╪═════════════╪════════╪══════════════════╡
│ Elderly   ┆ 1727        ┆ 234    ┆ 13.549508        │
│ Adult     ┆ 1866        ┆ 168    ┆ 9.003215         │
│ Pediatric ┆ 2           ┆ 0      ┆ 0.0              │
└───────────┴─────────────┴────────┴──────────────────┘

Community vs Hospital Acquired Sepsis:
shape: (3, 3)
┌──────────────────────┬─────────────┬──────────────────┐
│ Is_Hospital_Acquired ┆ Total_Cases ┆ Mortality_Rate_% │
│ ---                  ┆ ---         ┆ ---              │
│ bool                 ┆ u32         ┆ f64              │
╞══════════════════════╪═════════════╪══════════════════╡
│ false                ┆ 3221        ┆ 9.87271          │
│ null                 ┆ 18          ┆ 0.0              │
│ true                

/tmp/ipykernel_183242/4293122646.py:44: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("Total_Cases"),


In [67]:
df_all.filter(
    expr_culture_orders
)['event_full_name'].unique().to_list()

['Order - Procedure__CULTURE CSF + GRAM STAIN__Body Fluid Cultures',
 'Order - Procedure__CULTURE CYSTIC FIBROSIS SPUTUM__Body Fluid Cultures',
 'Order - Procedure__CULTURE BLOOD ID MALDI-ANAEROBIC__Body Fluid Cultures',
 'Order - Procedure__CULTURE BLOOD ID NAAT- AEROBIC__Body Fluid Cultures',
 'Order - Procedure__CULTURE SALMONELLA__Body Fluid Cultures',
 'Order - Procedure__CULTURE STERILE SITE__Body Fluid Cultures',
 'Order - Procedure__CULTURE CATH TIP__Body Fluid Cultures',
 'Order - Procedure__CULTURE RESPIRATORY (NON SPUTUM) + STAIN__Body Fluid Cultures',
 'Order - Procedure__CULTURE SPUTUM + SCREENING SMEAR__Body Fluid Cultures',
 'Order - Procedure__CULTURE GENITAL__Body Fluid Cultures',
 'Order - Procedure__CULTURE NOCARDIA WITH STAIN__Body Fluid Cultures',
 'Order - Procedure__CULTURE CAMPYLOBACTER__Body Fluid Cultures',
 'Order - Procedure__CULTURE CYSTIC FIBROSIS RESP (NON SPUTUM) + GRAM STAIN__Body Fluid Cultures',
 'Order - Procedure__CULTURE EYE EAR SINUS NP__Body Flui

In [56]:
df_insights

PAT_ENC_CSN_ID,Suspected_Infection_Time,Pt_Arrival,Admit_Time,Admitted_From_ED,Patient_Age,FLAG_DEATH_CLARITY,Sepsis_Category,Hours_Arrival_to_Suspicion,Is_Hospital_Acquired,Age_Group
i64,datetime[ns],datetime[ns],datetime[ns],str,f64,i64,str,i64,bool,str
727200338,2025-05-11 16:58:08,2025-05-11 15:46:00,2025-05-12 00:42:00,"""Yes""",75.997262,0,"""POA-1""",1,false,"""Elderly"""
714738435,2024-11-08 15:11:00,2024-10-29 10:32:50,2024-10-29 20:35:48,"""No""",56.862422,0,"""NPOA-3""",244,true,"""Adult"""
723798812,2025-03-22 00:11:37,2025-03-21 23:42:00,2025-03-22 15:03:00,"""Yes""",23.195071,0,"""POA-1""",0,false,"""Adult"""
728815638,2025-06-04 21:27:00,2025-06-04 19:05:00,2025-06-05 23:48:00,"""Yes""",55.137577,0,"""POA-1""",2,false,"""Adult"""
721712678,2025-02-18 17:10:22,2025-02-18 16:01:00,2025-02-18 22:27:00,"""Yes""",76.142368,0,"""POA-3""",1,false,"""Elderly"""
…,…,…,…,…,…,…,…,…,…,…
722086381,2025-02-24 22:07:15,2025-02-24 17:07:00,2025-02-24 21:23:00,"""Yes""",80.052019,0,"""NPOA-3""",5,false,"""Elderly"""
724661177,2025-04-04 05:53:00,2025-04-03 22:21:00,2025-04-04 17:35:00,"""Yes""",72.402464,0,"""POA-3""",7,false,"""Elderly"""
716116344,2024-11-18 12:33:40,2024-11-18 12:29:00,2024-11-19 22:06:00,"""Yes""",68.150581,0,"""POA-1""",0,false,"""Elderly"""


In [64]:
# COUNT THE # Of encounters in each group
df_insights.group_by(["Sepsis_Category", "FLAG_DEATH_CLARITY"]).agg(pl.col("Hours_Arrival_to_Suspicion").mean()).sort(by=["Sepsis_Category", "FLAG_DEATH_CLARITY"]).to_pandas()

,Sepsis_Category,FLAG_DEATH_CLARITY,Hours_Arrival_to_Suspicion
0,NPOA-1,0,143.893855
1,NPOA-1,1,91.615385
2,NPOA-2,0,94.447368
3,NPOA-2,1,41.750000
4,NPOA-3,0,134.538462
5,NPOA-3,1,111.604651
6,POA-1,0,8.447923
7,POA-1,1,25.741935
8,POA-2,0,6.987480
9,POA-2,1,4.500000


In [68]:
expr_clean_numeric = (
    pl.col("MEAS_VALUE")
    .str.extract(r"(-?\d+\.?\d*)", 1) # Regex to capture integer or float
    .cast(pl.Float64)
    .alias("val_numeric")
)
df_all.with_columns(
    expr_clean_numeric
).filter(pl.col("val_numeric").is_not_null())['event_full_name'].unique().to_list()

['Order Result - Lab__CREATININE POC__Creatinine',
 'Order Result - Lab__CULTURE BLOOD ID MALDI - ANAEROBIC__Body Fluid Cultures',
 'Order Result - Lab__CHLORIDE__Chloride',
 'Order Result - Lab__BUN POC__Blood Urea Nitrogen',
 'Order Result - Lab__AST__AST',
 'Flowsheet__BLOOD PRESSURE__Blood Pressure',
 'Order Result - Lab__SODIUM__Sodium',
 'Order Result - Microbiology__GRAM ST__Body Fluid Cultures',
 'Order Result - Lab__IMMATURE GRAN PCT__IMMATURE GRAN PCT',
 'Order Result - Lab__NRBC ABS__nRBC',
 'Order Result - Lab__MCHC__MCHC',
 'Flowsheet__R UROSTOMY OUTPUT__Urine Output',
 'Order Result - Lab__LYMPHS PCT__Lymphocytes Percent',
 'Flowsheet__TEMPERATURE__Temperature',
 'Order Result - Lab__CREATININE__Creatinine',
 'Flowsheet__CPM S24 R AS SC BRADEN SCORE__Braden Scale',
 'Order Result - Microbiology__CULTURE__Body Fluid Cultures',
 'Flowsheet__UTSW ANES R AN URINE OUTPUT__Urine Output',
 'Flowsheet__UTSW UH R UNMEASURED URINE OCCURENCE__Urine Output',
 'Order Result - Lab__CUL

In [77]:
df_all.with_columns(
    [
        pl.when(
        pl.col("EVENT_NAME")=="BLOOD PRESSURE"
        ).then(
            pl.col("MEAS_VALUE").str.split("/").list[0]
        ).otherwise(
            pl.lit(None)
        ).cast(pl.Float64).alias("Systolic_Pressure"),
        pl.when(
            pl.col("EVENT_NAME")=="BLOOD PRESSURE"
        ).then(
            pl.col("MEAS_VALUE").str.split("/").list[1]
        ).otherwise(
            pl.lit(None)
        ).cast(pl.Float64).alias("Diastolic_Pressure")
    ]
).with_columns(
    MAP = (pl.col("Systolic_Pressure") + 2*pl.col("Diastolic_Pressure"))/3.0
).select("MEAS_VALUE", "Systolic_Pressure", "Diastolic_Pressure", "MAP")

MEAS_VALUE,Systolic_Pressure,Diastolic_Pressure,MAP
str,f64,f64,f64
"""3840""",null,null,null
"""105/50""",105.0,50.0,68.333333
"""18""",null,null,null
"""76""",null,null,null
"""33.5""",null,null,null
…,…,…,…
"""Aerobic And An…",null,null,null
"""Aerobic And An…",null,null,null
"""Gram Positive …",null,null,null


In [ ]:
# 1. PREPARE THE RAW DATA (Lazy)
lf = df.lazy()

# -------------------------------------------------------------------------
# A. LABS (Platelets, Bilirubin, Creatinine)
# -------------------------------------------------------------------------
lf_labs = lf.filter(
    (pl.col("event_full_name").str.to_lowercase().str.contains("platelet|bilirubin|creatinine")) &
    (pl.col("event_full_name").str.to_lowercase().str.contains("result")) # Ensure it's a result
).select([
    "PAT_ENC_CSN_ID",
    "Event_DateTime",
    "event_full_name",
    expr_clean_numeric # Apply the cleaner here
]).filter(pl.col("val_numeric").is_not_null())

# Calculate Lab SOFA Points
lf_labs_scored = lf_labs.with_columns(
    pl.when(pl.col("event_full_name").str.to_lowercase().str.contains("platelet"))
      .then(
          pl.when(pl.col("val_numeric") < 20).then(4)
            .when(pl.col("val_numeric") < 50).then(3)
            .when(pl.col("val_numeric") < 100).then(2)
            .when(pl.col("val_numeric") < 150).then(1)
            .otherwise(0)
      )
      .when(pl.col("event_full_name").str.to_lowercase().str.contains("bilirubin"))
      .then(
          pl.when(pl.col("val_numeric") >= 12.0).then(4)
            .when(pl.col("val_numeric") >= 6.0).then(3)
            .when(pl.col("val_numeric") >= 2.0).then(2)
            .when(pl.col("val_numeric") >= 1.2).then(1)
            .otherwise(0)
      )
      .when(pl.col("event_full_name").str.to_lowercase().str.contains("creatinine"))
      .then(
          pl.when(pl.col("val_numeric") >= 5.0).then(4)
            .when(pl.col("val_numeric") >= 3.5).then(3)
            .when(pl.col("val_numeric") >= 2.0).then(2)
            .when(pl.col("val_numeric") >= 1.2).then(1)
            .otherwise(0)
      )
      .alias("SOFA_Points")
      .cast(pl.UInt8)
)

# -------------------------------------------------------------------------
# B. CARDIOVASCULAR (Calculating MAP from "BP")
# -------------------------------------------------------------------------
# We assume format "120/80". We need to split and calculate.
lf_bp = lf.filter(
    pl.col("event_full_name").str.to_lowercase().str.contains("blood pressure")
).select([
    "PAT_ENC_CSN_ID",
    "Event_DateTime",
    "MEAS_VALUE"
]).with_columns(
    # Split "120/80" into two columns
    pl.col("MEAS_VALUE").str.split("/").arr.get(0).cast(pl.Float64, strict=False).alias("Sys"),
    pl.col("MEAS_VALUE").str.split("/").arr.get(1).cast(pl.Float64, strict=False).alias("Dia")
).filter(
    pl.col("Sys").is_not_null() & pl.col("Dia").is_not_null()
).with_columns(
    # Calculate Mean Arterial Pressure (MAP)
    ((pl.col("Sys") + (2 * pl.col("Dia"))) / 3).alias("val_numeric")
).with_columns(
    # Assign Score (Simple MAP < 70 check)
    # Note: Vasopressors override this score, but this is the baseline check.
    pl.when(pl.col("val_numeric") < 70).then(1).otherwise(0).alias("SOFA_Points")
)